# Time Series Analysis: Cross-Validation Strategies & Forecasting Models

Time series data represents sequences of data points indexed in successive order over time. Unlike standard tabular datasets where observations are assumed to be Independent and Identically Distributed (I.I.D.), time series data exhibits temporal dependency, autocorrelation, and non-stationarity.

This notebook covers two fundamental pillars of time series engineering:
1. **Cross-Validation Strategies:** Preventing data leakage and look-ahead bias during model evaluation.
2. **Forecasting Models & Ecosystem:** A comprehensive breakdown of classical statistical models (ARMA, ARIMA, SARIMA), additive models (Prophet), and modern machine learning/deep learning approaches.

## Part 1: Cross-Validation in Time Series

### Why Standard K-Fold Cross-Validation Fails
In standard $K$-Fold cross-validation, data points are randomly shuffled into folds. When applied to time series, this creates catastrophic problems:

* **Look-Ahead Bias (Data Leakage):** The model trains on future data points to predict past data points (e.g., using stock prices from Friday to predict Wednesday).
* **Autocorrelation Disruption:** Time series observations are dependent on their immediate predecessors. Shuffling destroys this sequential structure.
* **Overly Optimistic Metrics:** Because the model learns patterns from future data that would never be available in a real deployment, evaluation scores become unrealistically high.

To evaluate time series models properly, validation splits must strictly respect the chronological arrow of time.

### 1. Expanding Window (Forward Chaining / Walk-Forward Validation)

This is the standard and most widely used cross-validation strategy for time series (implemented as `TimeSeriesSplit` in `scikit-learn`).

The model trains on an initial historical window and predicts the immediate next horizon. With each subsequent fold, the training window **expands** forward in time while the test set shifts ahead.

```text
Fold 1: [ Train ] [ Test ]
Fold 2: [    Train    ] [ Test ]
Fold 3: [       Train       ] [ Test ]
Fold 4: [          Train          ] [ Test ]
───► Time Flow
```

* **How it works:**
  * Fold 1: Train on $[t_0 \dots t_k]$, Test on $[t_{k+1} \dots t_{k+m}]$
  * Fold 2: Train on $[t_0 \dots t_{k+m}]$, Test on $[t_{k+m+1} \dots t_{k+2m}]$
  * Fold 3: Train on $[t_0 \dots t_{k+2m}]$, Test on $[t_{k+2m+1} \dots t_{k+3m}]$
* **Best used for:** Problems where the entire historical record remains informative (e.g., long-term economic trends, seasonal cycles across multiple years).
* **Scikit-Learn Implementation:** `from sklearn.model_selection import TimeSeriesSplit`

### 2. Rolling Window (Sliding Window)

Instead of expanding the training set indefinitely, the training set maintains a **fixed window size** that slides forward through time along with the test set. Older historical data is systematically dropped as new data arrives.

```text
Fold 1: [ Train ] [ Test ]
Fold 2:   [ Train ] [ Test ]
Fold 3:     [ Train ] [ Test ]
Fold 4:       [ Train ] [ Test ]
───► Time Flow
```

* **How it works:**
  * Fold 1: Train on $[t_0 \dots t_k]$, Test on $[t_{k+1} \dots t_{k+m}]$
  * Fold 2: Train on $[t_m \dots t_{k+m}]$, Test on $[t_{k+m+1} \dots t_{k+2m}]$
  * Fold 3: Train on $[t_{2m} \dots t_{k+2m}]$, Test on $[t_{k+2m+1} \dots t_{k+3m}]$
* **Best used for:** Non-stationary data with **concept drift** or regime shifts (e.g., high-frequency financial trading, rapidly changing user behavior) where data from several years ago is obsolete or misleading.

### 3. Purged and Embargoed Cross-Validation

Developed by Marcos López de Prado for financial and quantitative applications.

When target labels span multiple time periods into the future (e.g., a 5-day forward return or rolling volatility), standard temporal splits can still leak information if a training label's calculation window overlaps with the test window.

```text
Fold: [ Train ] [ Gap / Purge ] [ Test ] [ Embargo Gap ] [ Train ]
```

* **Purging:** Removes training observations whose evaluation period overlaps with the test window.
* **Embargoing:** Inserts a buffer period ("embargo") immediately after the test set before the next training fold begins, preventing post-test autoregressive spillover.
* **Best used for:** High-frequency trading, quantitative finance, or any domain with overlapping label windows and multi-step lookaheads.

### 4. Grouped / Blocked Time Series Split (Panel Data)

When the dataset contains multiple independent series tracked across the same timeline (e.g., sales across 100 retail stores, or telemetry from 500 IoT sensors):

* **The Rule:** Always split strictly across the **time axis for all groups simultaneously**.
* **Do Not:** Split randomly by row or by entity, as that leaks concurrent information across entities at identical timestamps.
* **Example:** Train on all 100 stores from January through August; evaluate on all 100 stores from September through December.

## Part 2: Time Series Forecasting Models & Terminology

Time series models are categorized based on their underlying mathematical approach: classical statistical models, additive curve-fitting models, and modern machine learning/deep learning architectures.

### 1. The Classical Statistical Family: AR, MA, ARMA, ARIMA, SARIMA

#### **AR (AutoRegressive) Model**
* Forecasts the variable using a linear combination of its own past (lagged) values.
* **Concept:** *"Today's value is a weighted sum of yesterday's and the day before yesterday's values plus noise."*
* **Formula:** $Y_t = c + \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + \dots + \epsilon_t$

#### **MA (Moving Average) Model**
* Forecasts the variable using past forecast errors (shocks or white noise).
* **Concept:** *"Today's value adjusts based on unexpected surprises or residual errors from previous forecasts."*
* **Formula:** $Y_t = \mu + \epsilon_t + \theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2} + \dots$

#### **ARMA (AutoRegressive Moving Average)**
* Combines both AutoRegressive (AR) and Moving Average (MA) mechanisms.
* **Core Requirement:** ARMA requires the data to be strictly **stationary** (constant mean, constant variance, and time-invariant autocorrelation). It cannot handle data with trends.

#### **ARIMA (AutoRegressive Integrated Moving Average)**
* Adds the letter **I** for **"Integrated"**.
* **What Integrated means:** If a time series has a trend (is non-stationary), ARIMA computes differences between consecutive observations ($Y_t - Y_{t-1}$) to stabilize the mean before applying ARMA.
* **Notation:** $\text{ARIMA}(p, d, q)$
  * $p$: Lag order (AR part)
  * $d$: Degree of differencing required to achieve stationarity (I part)
  * $q$: Window size of the moving average (MA part)

#### **SARIMA (Seasonal ARIMA)**
* Adds the letter **S** for **"Seasonal"**.
* Standard ARIMA handles non-seasonal trends, but struggles when patterns repeat at fixed intervals (e.g., peak electricity demand every evening, or retail sales surges every December).
* SARIMA adds seasonal differencing and seasonal AR/MA terms.
* **Notation:** $\text{SARIMA}(p, d, q)(P, D, Q)_s$
  * $(p, d, q)$: Non-seasonal orders
  * $(P, D, Q)$: Seasonal orders
  * $s$: Length of the seasonal cycle (e.g., $s=12$ for monthly data with annual seasonality, $s=7$ for daily data with weekly seasonality).

### 2. Prophet (By Meta / Facebook)

Developed by Facebook's Core Data Science team in 2017, **Prophet** was designed for business forecasting where data often exhibits multiple seasonal patterns, holidays, and trend shifts.

Unlike ARIMA (which models step-to-step autocorrelation), Prophet treats forecasting as a **curve-fitting / generalized additive model**:

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

* $g(t)$: **Trend function** — models non-periodic changes using piecewise linear or logistic growth curves, automatically detecting changepoints where growth rates shift.
* $s(t)$: **Seasonality** — models periodic changes using Fourier series (handles daily, weekly, and yearly seasonality simultaneously).
* $h(t)$: **Holiday effects** — incorporates user-defined or calendar-specific irregular events (e.g., Super Bowl, Black Friday, Diwali).
* $\epsilon_t$: Error term (normally distributed noise).

#### Key Advantages of Prophet
* Automatically accounts for multiple seasonalities simultaneously.
* Robust to missing observations and large outliers.
* Does not require manual differencing or stationarity checks.
* Highly interpretable: components (trend, weekly, yearly, holidays) can be plotted and inspected individually.

### 3. Other Key Time Series Models & Acronyms

#### Extensions of ARIMA
* **ARIMAX / SARIMAX:** The **X** stands for **eXogenous variables**. Incorporates external explanatory features (e.g., predicting retail sales using past sales *plus* temperature, marketing spend, and discount rates).
* **Auto-ARIMA:** An automated algorithm (available via `pmdarima` in Python) that conducts a stepwise grid search to discover the optimal $(p, d, q)(P, D, Q)_s$ parameters using AIC/BIC minimization.

#### Multivariate Models (Multiple Interdependent Series)
* **VAR (Vector AutoRegression):** Extends AR to multiple time series where variables bidirectionally influence each other (e.g., simultaneously modeling inflation rate, unemployment, and interest rate).
* **VARMA / VARMAX:** Vector ARMA with exogenous regressors.
* **VECM (Vector Error Correction Model):** Used for non-stationary multivariate series that share a long-term equilibrium relationship (cointegration).

#### Exponential Smoothing & State-Space Models
* **ETS (Error, Trend, Seasonal):** A state-space framework generalizing Exponential Smoothing. Weighs past observations with exponentially decaying weights.
* **Holt-Winters:** A classic three-parameter exponential smoothing method for level, trend, and seasonality.
* **TBATS:** Accommodates complex, high-frequency, non-integer seasonalities (Trigonometric, Box-Cox, ARMA errors, Trend, Seasonal).

#### Financial Volatility Models
* **ARCH (AutoRegressive Conditional Heteroskedasticity):** Models changes in variance/volatility over time rather than predicting the series value itself.
* **GARCH (Generalized ARCH):** The industry standard for financial quants to model risk, asset price volatility clustering, and Value-at-Risk (VaR).

#### Modern Deep Learning Forecasting
* **DeepAR:** Amazon's probabilistic RNN/LSTM model designed to train across thousands of related series simultaneously (e.g., demand forecasting for millions of retail catalog items).
* **N-BEATS / N-HiTS:** Pure deep learning architectures based on backward and forward residual links for fast, interpretable trend/seasonality decomposition without recurrence.
* **TFT (Temporal Fusion Transformer):** Attention-based architecture designed for multi-horizon forecasting, combining high performance with interpretable variable selection and attention weights.
* **PatchTST:** State-of-the-art transformer model that divides continuous time series into segmented "patches" (sub-series tokens).

### 4. Summary: Model Selection Guide

| Model Family | Examples | Best Suited For | Key Strength |
| :--- | :--- | :--- | :--- |
| **Classical Univariate** | ARMA, ARIMA, SARIMA | Single series with clear autocorrelations | Strong statistical foundation, minimal data needed |
| **Univariate + External Features** | SARIMAX | Demand/sales with known covariates (pricing, weather) | Combines time series memory with regression features |
| **Multivariate** | VAR, VARMAX, VECM | Interdependent systems (macroeconomics, finance) | Captures mutual feedback loops between series |
| **Additive / Curve Fitting** | Prophet | Business KPIs, revenue, traffic with multiple cycles | Handles multiple seasonalities, holidays, missing data |
| **Volatility / Risk** | ARCH, GARCH | Stock returns, foreign exchange volatility | Models variance clustering and financial risk |
| **Deep Learning** | DeepAR, TFT, PatchTST | Large fleets of related series (e.g., e-commerce) | Learns cross-series patterns and complex non-linearities |